# Document Ingestion Testing

In [1]:
# packages

## link project directory
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

## custom packages
from ingestion.documents.sec import SecEdgarProvider

## data collection
import sqlite3
import pandas as pd

## other
from tqdm import tqdm

In [2]:
# constants
from constants import (
    DEFAULT_SQLITE_PATH,
    USER_AGENT, FILING_TYPES, START_DATE, END_DATE
)

In [3]:
# connect to data
connection = sqlite3.connect(DEFAULT_SQLITE_PATH)
connection.row_factory = sqlite3.Row

# retrieve CIKs
query = "select distinct company_id, cik from companies"
ciks = pd.read_sql_query(query, connection).set_index('company_id').to_dict()['cik']


## Retrieve Raw HTML

In [4]:
provider = SecEdgarProvider(user_agent =  USER_AGENT)
filings = provider.list_filings(
    cik = ciks['GOOG'], 
    filing_types = FILING_TYPES, 
    start_date = START_DATE, end_date = END_DATE
)
for filing in filings:
    print(
        filing.filing_type,
        filing.filing_date,
        filing.accession_number
    )

10-Q 2026-04-30 0001652044-26-000048
10-K 2026-02-05 0001652044-26-000018
10-Q 2025-10-30 0001652044-25-000091
10-Q 2025-07-24 0001652044-25-000062
10-Q 2025-04-25 0001652044-25-000043
10-K 2025-02-05 0001652044-25-000014
10-Q 2024-10-30 0001652044-24-000118
10-Q 2024-07-24 0001652044-24-000079
10-Q 2024-04-26 0001652044-24-000053
10-K 2024-01-31 0001652044-24-000022
10-Q 2023-10-25 0001652044-23-000094
10-Q 2023-07-26 0001652044-23-000070


In [5]:
# iterate over the CIKs and get all filings

# define the provider
provider = SecEdgarProvider(user_agent =  USER_AGENT)

# iterate through each company to get the filings
# ciks = {'GOOG': <cik>, 'AAPL': <cik>, ...}
seen = set() # track seen CIKs to avoid duplicates (i.e., GOOG, GOOGL)
# for ticker, cik in tqdm(
#     ciks.items(),
#     desc = 'Companies',
#     unit = 'company'
# ):
for ticker, cik in ciks.items():
    # check if we have already seen this CIK (i.e., GOOG, GOOGL)
    if cik in seen:
        continue
    seen.add(cik)

    # get the filings
    filings = provider.list_filings(
        cik = cik,
        filing_types = FILING_TYPES,
        start_date = START_DATE,
        end_date = END_DATE
    )

    # store each of the retrieved HTML documents
    for f in tqdm(filings, desc = f'[{ticker}] Downloading filings', unit = 'filing'):
        # download the filing (or understand why failure occurred)
        try:
            html_doc = provider.download_filing(f)
        except Exception as e:
            print(f'Failed: {ticker} {f.accession_number}: {e}')
            continue
        
        # store the document in the database

        ## define the filepath
        fdate = f.filing_date.strftime("%Y-%m-%d")
        fname = f'{fdate}_{f.filing_type}_{f.accession_number}'
        fpath = PROJECT_ROOT / 'data' / 'raw' / 'sec_edgar' / ticker / f'{fname}.html'
        if fpath.exists():
            continue # skip if the file already exists

        ## write the file
        fpath.parent.mkdir(parents = True, exist_ok = True)
        with open(fpath, 'w', encoding='utf-8') as f:
            f.write(html_doc)

[TSLA] Downloading filings: 100%|██████████| 26/26 [00:01<00:00, 19.65filing/s]


In [ ]:
# load test file
from bs4 import BeautifulSoup

test_path = (
    PROJECT_ROOT / 'data' / 'raw' / 'sec_edgar' / 'GOOG' / 
    '2023-07-26_10-Q_0001652044-23-000070.html'
)

with open(test_path, 'r', encoding = 'utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

In [38]:
# get testing filing metadata
provider = SecEdgarProvider(user_agent =  USER_AGENT)
test_filing = provider.list_filings(
    cik = ciks['GOOG'],
    filing_types = FILING_TYPES,
    start_date = '2023-07-01',
    end_date = '2023-08-01'
)[0]

In [ ]:
# src / ingestion / documents / parser.py
from pathlib import Path
from bs4 import BeautifulSoup, Comment

from ingestion.documents.sec import FilingMetadata

class DocumentMetadata:
    company: str
    ticker: str
    cik: str
    filing_type: str
    filing_date: str
    period_start: str 
    period_end: str 
    accession_number: str
    source: str 
    raw_html_path: str 

class Paragraph:
    text: str 

class Table:
    caption: str
    headers: str
    rows: list[list[str]]

class Section:
    title: str
    blocks: list[Paragraph | Table]

class Document:
    metadata: DocumentMetadata
    sections: list[Section]

class DocumentParser:
    """ 
    Description
    ----------
    This class contains the methods necessary to restructure an HTML
    file into a highly structured text class ready for downstream
    RAG processing.

    NOTE
    - self.metadata.period_start and self.metadata.period_end may have
      variable date formatting.
    """
    def __init__(self):
        super().__init__()

        # instantiate objects to be created later
        self.html_path = None
        self.soup = None
        self.metadata = DocumentMetadata()

    # =======================
    # === Public Methods  ===
    # =======================

    def load_html(self, html_path: str):
        """
        Description
        ----------
        Loads the html file from the given path and returns a BeautifulSoup object.

        Inputs
        ----------
        html_path = The path to the HTML file to be loaded

        Returns
        ----------
        BeautifulSoup object of the loaded HTML file stored in self.soup
        """
        # store the path for later use
        self.html_path = Path(html_path)

        # read the html
        with open(html_path, 'r', encoding = 'utf-8') as f:
            html = f.read()

        self.soup = BeautifulSoup(html, 'html.parser')

        # store metadata attributes before proceeding to cleaning

        ## company name
        self.metadata.company = self._find_ix_value('dei:EntityRegistrantName')

        ## period reported
        context = self.soup.find("xbrli:context")
        start = self._find_ix_value("dei:DocumentPeriodStartDate")
        if start is not None:
            self.metadata.period_start = start 
        else:
            self.metadata.period_start = context.find("xbrli:startdate").get_text(strip = True)
            
        end = self._find_ix_value("dei:DocumentPeriodEndDate")
        if end is not None:
            self.metadata.period_end = end 
        else:
            self.metadata.period_end = context.find("xbrli:enddate").get_text(strip = True)

    def clean_dom(self):
        """
        Description
        ----------
        Removes the HTML elements that do not pertain to the semantic content
        of the document.

        NOTE: This method only removes things that are always safe to remove.

        Inputs
        ----------
        None

        Returns
        ----------
        None. self.soup is modified in place.
        """
        if self.soup is None:
            raise ValueError("HTML document not loaded. Please call load_html() first.")

        # --- Remove Javascript and CSS ---
        for tag in self.soup.find_all(['script', 'style']):
            tag.decompose()

        # --- Remove HTML comments ---
        for comment in self.soup.find_all(string=lambda text: isinstance(text, Comment)):
            comment.decompose()

        # --- Remove hidden elements ---
        for tag in self.soup.find_all(style = True):
            style = tag['style'].replace(' ', '').lower()
            if (
                'display:none' in style or 
                'visibility:hidden' in style or 
                'opacity:0' in style
            ):
                tag.decompose()

        # --- Remove anchor tags, but preserve content ---
        for tag in self.soup.find_all('a'):
            tag.unwrap()

        # --- Remove inline XBRL wrappers while preserving text ---
        for tag in self.soup.find_all():
            if ':' in tag.name:
                prefix = tag.name.split(":")[0]
                if prefix in {'ix', 'ixt', 'ixt-sec'}:
                    tag.unwrap()

        # --- Remove empty tags ---
        for tag in self.soup.find_all():
            if (
                tag.name not in {'br', 'hr'} and 
                not tag.get_text(strip = True) and 
                not tag.find('table')
            ):
                tag.decompose()

    def extract_metadata(
        self,
        filing_metadata: FilingMetadata,
        ticker: str,
        source: str = 'SEC EDGAR'
    ):
        """
        Description
        ----------
        This method builds the metadata for the document based on
        previously known information within the production pipeline
        and information we must extract from the self.soup

        Inputs
        ----------
        filing_metadata = The FilingMetadata object created as part of 
            retrieving the document from the SEC EDGAR database
        ticker = The ticker for the company
        source = The source of the data. Defaults to 'SEC EDGAR'
        """
        # --- populate metadata with already known info ---

        ## define attributes already known by DocumentParser
        self.metadata.raw_html_path = self.html_path
        
        ## define attributes already known from the filing
        self.metadata.cik = filing_metadata.cik
        self.metadata.accession_number = filing_metadata.accession_number
        self.metadata.filing_type = filing_metadata.filing_type
        self.metadata.filing_date = filing_metadata.filing_date

        ## define attributes known other provided params
        self.metadata.ticker = ticker
        self.metadata.source = source

        # --- extract the rest of the info needed for metadata ---

    def remove_boilerplate(self):
        raise NotImplementedError

    def html_to_blocks(self):
        raise NotImplementedError

    def split_into_sections(self):
        raise NotImplementedError

    def build_document(self):
        raise NotImplementedError

    # =======================
    # === Private Methods ===
    # =======================

    def _find_ix_value(self, name: str) -> str | None:
        tag = self.soup.find(attrs = {'name': name})

        if tag is None:
            return None 

        return tag.get_text(strip = True)

dp = DocumentParser()
dp.load_html(test_path)
dp.clean_dom()
dp.extract_metadata(
    filing_metadata = test_filing,
    ticker = 'GOOG'
)

print(f'{dp.__class__.__name__} Attributes')
for k, v in vars(dp.metadata).items():
    print(f'\t{k}: {v}')

# print(dp.soup.prettify()[-3000:])

DocumentParser Attributes
	company: Alphabet Inc.
	period_start: 2023-01-01
	period_end: June 30, 2023
	raw_html_path: /Users/nickcruickshank/Projects/ai-investment-decision-support/data/raw/sec_edgar/GOOG/2023-07-26_10-Q_0001652044-23-000070.html
	cik: 0001652044
	accession_number: 0001652044-23-000070
	filing_type: 10-Q
	filing_date: 2023-07-26
	ticker: GOOG
	source: SEC EDGAR
